In [2]:
import time
import requests
import pandas as pd
from pathlib import Path

CSV_PATH = "filing_data/sp500.csv"
OUTPUT_DIR = "sec_filings_100"
FORM_TYPE = "10-K"

HEADERS = {
    "User-Agent": "Arnav Singh arnav2003.singh@gmail.com"
}

df = pd.read_csv(CSV_PATH)
df["CIK"] = df["CIK"].astype(str).str.zfill(10)

Path(OUTPUT_DIR).mkdir(exist_ok=True)

for _, row in df.iterrows():
    ticker = row["Symbol"]
    cik = row["CIK"]

    try:
        print(f"Downloading {FORM_TYPE} for {ticker}")

        submissions_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
        submissions = requests.get(submissions_url, headers=HEADERS)
        submissions.raise_for_status()
        data = submissions.json()

        recent = data["filings"]["recent"]

        found = False

        for i, form in enumerate(recent["form"]):
            if form == FORM_TYPE:
                accession = recent["accessionNumber"][i]
                accession_clean = accession.replace("-", "")
                primary_doc = recent["primaryDocument"][i]
                filing_date = recent["filingDate"][i]

                filing_url = (
                    f"https://www.sec.gov/Archives/edgar/data/"
                    f"{int(cik)}/{accession_clean}/{primary_doc}"
                )

                response = requests.get(filing_url, headers=HEADERS)
                response.raise_for_status()

                filename = f"{ticker}_{FORM_TYPE}_{filing_date}.html"
                save_path = Path(OUTPUT_DIR) / filename

                with open(save_path, "w", encoding="utf-8") as f:
                    f.write(response.text)

                print(f"Saved {filename}")
                found = True
                break

        if not found:
            print(f"No {FORM_TYPE} found for {ticker}")

        time.sleep(0.2)

    except Exception as e:
        print(f"Error for {ticker}: {e}")

print("Done.")

Saved GOOGL_10-K_2026-02-05.html
Saved T_10-K_2026-02-09.html
Saved EA_10-K_2026-05-11.html
Saved META_10-K_2026-01-29.html
Saved NFLX_10-K_2026-01-23.html
Saved OMC_10-K_2026-02-20.html
Saved TTWO_10-K_2026-05-22.html
Saved DIS_10-K_2025-11-13.html
Saved WBD_10-K_2026-02-27.html
Saved ABNB_10-K_2026-02-12.html
Saved AMZN_10-K_2026-02-06.html
Saved BKNG_10-K_2026-02-18.html
Saved DASH_10-K_2026-02-18.html
Saved NKE_10-K_2025-07-17.html
Saved RL_10-K_2026-05-21.html
Saved SBUX_10-K_2025-11-14.html
Saved TPR_10-K_2025-08-14.html
Saved TSLA_10-K_2026-01-29.html
Saved CASY_10-K_2026-06-22.html
Saved KO_10-K_2026-02-20.html
Saved CL_10-K_2026-02-23.html
Saved COST_10-K_2025-10-08.html
Saved DG_10-K_2026-03-20.html
Saved GIS_10-K_2026-07-01.html
Saved PEP_10-K_2026-02-03.html
Saved TGT_10-K_2026-03-11.html
Saved WMT_10-K_2026-03-13.html
Saved APA_10-K_2026-02-26.html
Saved BKR_10-K_2026-02-05.html
Saved CVX_10-K_2026-02-24.html
Saved COP_10-K_2026-02-17.html
Saved DVN_10-K_2026-02-18.html
Sa